# Extending Acquisition Functions

This tutorial shows how to create custom acquisition functions by extending [`AcquisitionFunction`](https://instadeepai.github.io/alf/api/alf_core/optimizer/acquisition_function/). We'll implement uncertainty sampling and diversity-based acquisition as examples.

## 1. Imports

In [ ]:
import numpy as np
from alf_core.dataclasses import Candidate, LabelledCandidates, TaskState
from alf_core.optimizer.acquisition_function import AcquisitionFunction

## 2. Define Custom Acquisition Functions

Implement the `__call__()` method to score candidates based on your acquisition strategy. The method receives a list of [`Candidate`](https://instadeepai.github.io/alf/api/alf_core/dataclasses/candidate/) objects and a [`TaskState`](https://instadeepai.github.io/alf/api/alf_core/dataclasses/task_state/) containing the surrogate model and dataset. It must return [`LabelledCandidates`](https://instadeepai.github.io/alf/api/alf_core/dataclasses/labelled_candidates/) where labels represent acquisition scores.

In [ ]:
class UncertaintySampling(AcquisitionFunction):
    """Select candidates with highest prediction uncertainty."""

    def __call__(
        self,
        search_candidates: list[Candidate],
        state: TaskState,
    ) -> LabelledCandidates:
        """Score candidates by their prediction variance."""
        # Get predictions from surrogate model
        predictions = state.surrogate.predict(search_candidates)

        # Use variance as acquisition score (higher = more uncertain)
        if predictions.variances is not None:
            scores = predictions.variances
        else:
            # Fallback if no variance available
            scores = np.ones(len(search_candidates))

        return LabelledCandidates(candidates=search_candidates, labels=scores)


class UpperConfidenceBound(AcquisitionFunction):
    """UCB acquisition: balance exploitation and exploration.

    β (beta) is the exploration parameter that controls how much uncertainty
    is weighted when selecting actions. It scales the confidence interval term
    added to the estimated value, balancing exploration vs. exploitation:
    higher β → more exploration; lower β → more exploitation.
    """

    def __init__(self, beta: float = 2.0):
        self.beta = beta

    def __call__(
        self,
        search_candidates: list[Candidate],
        state: TaskState,
    ) -> LabelledCandidates:
        """Score candidates using UCB: mean + beta * std."""
        predictions = state.surrogate.predict(search_candidates)

        # UCB formula
        means = predictions.means
        if predictions.variances is not None:
            stds = np.sqrt(predictions.variances)
            scores = means + self.beta * stds
        else:
            scores = means

        return LabelledCandidates(candidates=search_candidates, labels=scores)


class DiversitySampling(AcquisitionFunction):
    """Select diverse candidates based on distance from training data."""

    def __call__(
        self,
        search_candidates: list[Candidate],
        state: TaskState,
    ) -> LabelledCandidates:
        """Score candidates by minimum distance to training set."""
        # Extract candidate data
        candidate_data = np.array([c.data for c in search_candidates])
        training_data = np.array([c.data for c in state.dataset.candidates])

        # Compute minimum distance to any training point
        scores = []
        for point in candidate_data:
            distances = np.abs(training_data - point)
            min_distance = np.min(distances) if len(distances) > 0 else 1.0
            scores.append(min_distance)

        scores = np.array(scores)
        return LabelledCandidates(candidates=search_candidates, labels=scores)

## 3. Usage Example

In [ ]:
from alf_core.dataclasses import Modality, Predictions


# Create mock surrogate that returns predictions
class MockSurrogate:
    def predict(self, candidates):
        n = len(candidates)
        means = np.random.rand(n)
        variances = np.random.rand(n) * 0.1
        return Predictions(means=means, variances=variances)


# Create mock task state
training_candidates = [Candidate(data=x, modality=Modality.TABULAR) for x in [1.0, 3.0, 5.0]]
training_labels = np.array([0.2, 0.6, 0.9])
dataset = LabelledCandidates(candidates=training_candidates, labels=training_labels)

task_state = TaskState(dataset=dataset, surrogate=MockSurrogate(), round=1, acq_batch_size=5)

# Generate search candidates
search_candidates = [Candidate(data=x, modality=Modality.TABULAR) for x in np.linspace(0, 10, 20)]

# Apply different acquisition functions
print("=== Uncertainty Sampling ===")
uncertainty_acq = UncertaintySampling()
scored = uncertainty_acq(search_candidates, task_state)
top_5 = scored.get_top_k(5)
print(f"Top 5 scores: {top_5.labels}")
print(f"Top 5 candidates: {[c.data for c in top_5.candidates]}")

print("\n=== Upper Confidence Bound ===")
ucb_acq = UpperConfidenceBound(beta=2.0)
scored = ucb_acq(search_candidates, task_state)
top_5 = scored.get_top_k(5)
print(f"Top 5 scores: {top_5.labels}")

print("\n=== Diversity Sampling ===")
diversity_acq = DiversitySampling()
scored = diversity_acq(search_candidates, task_state)
top_5 = scored.get_top_k(5)
print(f"Top 5 scores: {top_5.labels}")
print(f"Top 5 candidates: {[c.data for c in top_5.candidates]}")

## Key Points

- **Required method**: `__call__(search_candidates, state)` must return `LabelledCandidates`
- **Input**: Receives unlabelled candidates from search function and current task state
- **Output**: Return candidates with acquisition scores as labels (higher = better)
- **Surrogate access**: Use `state.surrogate.predict()` to get predictions
- **Training data**: Access via `state.dataset` to avoid re-selecting points
- **Scoring**: Higher scores indicate candidates worth evaluating
- **Selection**: Framework automatically selects top-k candidates based on scores

Common acquisition strategies:
- **Greedy**: Select highest predicted value (exploitation)
- **Uncertainty**: Select most uncertain predictions (pure exploration)
- **UCB**: Balance exploitation and exploration with beta parameter
- **Expected Improvement**: Probability of improvement over current best
- **Thompson Sampling**: Sample from posterior distribution
- **Diversity**: Maximize distance from existing points

See existing implementations in `tools/alf_tools/optimizer/acquisition_functions/` for more examples.